# 08_4_3 DQA-SBA short plateau policy

08_4_2の評価では、`scene_total` が warmup `mAP50=0.353 / mAP50-95=0.189`、Phase1 r003 `0.380 / 0.206`、Phase1 r006 `0.380 / 0.205`、Phase2 r001 `0.375 / 0.207`、Phase2 r012 `0.356 / 0.194` だった。

Phase1 DQA-SBA自体は効いているが、Phase2は抑えてもroundが増えるほど pseudo box の平均confidence/objectnessが上がり、mAPが落ちる。つまり「品質が上がっている」のではなく、モデルが拾いやすいboxへ分布が寄っている可能性が高い。

08_4_3では、GTでbest checkpointを保持するのではなく、設計として更新量を短期でplateauさせる。

- Phase 1: DQA-SBAを6 roundだけ実行し、08_4_2で良かった r3-r6 付近に寄せる
- Phase 2: 16 round回す。ただし最初の3 roundだけをadapt impulseにし、その後は residual / classwise blend / pseudo loss / LR をほぼゼロへ落とす
- 目的: roundを増やしても性能が崩れず、一定値に収束するかを見る

実測では08_4_2が約3時間10分だった。今回は warmup 20 epoch + Phase1 6 round + Phase2 16 round + 評価で、全体4時間以内を目標にする。


In [ ]:
from __future__ import annotations

import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    required = (
        "dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase1_dqa_sba_short_plateau_policy.py",
        "dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py",
        "navigating_data_heterogeneity/setup_fedsto_scene_reproduction.py",
    )
    for base in (start, *start.parents):
        for candidate in (base, base / "Object_Detection"):
            if all((candidate / marker).exists() for marker in required):
                return candidate.resolve()
    raise FileNotFoundError("Could not locate /app/Object_Detection")


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase1_dqa_sba_short_plateau_policy.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
POLICY_MODEL = DQA_ROOT / "threshold_policy_model" / "artifacts" / "dqa05_threshold_policy.joblib"

WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_4_3_scene_phase1_dqa_sba_short_plateau_policy"
STATS_ROOT = DQA_ROOT / "stats_dqa08_4_3_scene_phase1_dqa_sba_short_plateau_policy"
LOG_ROOT = DQA_ROOT / "logs_dqa08_4_3_scene_phase1_dqa_sba_short_plateau_policy"
RUNNER_LOG = LOG_ROOT / "runner.out"
TRAIN_LOG = LOG_ROOT / "train.log"
PID_PATH = LOG_ROOT / "runner.pid"
THRESHOLD_LOG = STATS_ROOT / "phase1_dqa_sba_short_plateau_policy_schedule.jsonl"

preferred_python = Path("/root/micromamba/envs/al_yolov8/bin/python")
PYTHON_BIN = preferred_python if preferred_python.exists() else Path(sys.executable)

for path in [WORK_ROOT, STATS_ROOT, LOG_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("repo_root:", REPO_ROOT)
print("run_script:", RUN_SCRIPT, RUN_SCRIPT.exists())
print("workspace:", WORK_ROOT)
print("stats_root:", STATS_ROOT)
print("python:", PYTHON_BIN)
print("policy_model:", POLICY_MODEL, POLICY_MODEL.exists())
print("threshold_log:", THRESHOLD_LOG)


In [ ]:
# Full pipeline schedule.  08_4_3 keeps Phase 1 DQA-SBA and makes Phase 2 a short impulse then near-freeze plateau.
WARMUP_EPOCHS = 20
PHASE1_ROUNDS = 6
PHASE2_ROUNDS = 16
DQA_START_PHASE = 1

BATCH_SIZE = 160
WORKERS = 8
REQUESTED_GPUS = 2
MIN_FREE_GIB = 8

# Phase 1 DQA-SBA: same start as 08_4, faster residual decay after the early useful rounds.
PHASE1_RESIDUAL_START = 0.42
PHASE1_RESIDUAL_END = 0.12
PHASE1_MAX_RELATIVE_UPDATE = 0.025
PHASE1_SERVER_ANCHOR = 1.25
PHASE1_TEMPERATURE = 2.0
PHASE1_STABILITY_LAMBDA = 0.35
PHASE1_UNIFORM_MIX = 0.05

# Phase 2 base config.  The runner further schedules these into a decaying plateau policy.
CLASSWISE_BLEND = 0.040
SERVER_ANCHOR = 24.0
TEMPERATURE = 3.4
STABILITY_LAMBDA = 0.82

# Phase 2 short-plateau policy: adapt only in the first few rounds, then keep moving at near-zero strength.
PHASE2_RESIDUAL_START = 0.040
PHASE2_RESIDUAL_END = 0.002
PHASE2_CLASSWISE_BLEND_START = 0.040
PHASE2_CLASSWISE_BLEND_END = 0.003
PHASE2_MIN_SERVER_ALPHA_START = 0.88
PHASE2_MIN_SERVER_ALPHA_END = 0.985
PHASE2_SERVER_ANCHOR_START = 24.0
PHASE2_SERVER_ANCHOR_END = 80.0
PHASE2_TEMPERATURE_START = 3.4
PHASE2_TEMPERATURE_END = 6.0
PHASE2_STABILITY_LAMBDA_START = 0.82
PHASE2_STABILITY_LAMBDA_END = 0.95
PHASE2_DECAY_ROUNDS = 3
PHASE2_CLIENT_TRAIN_SCOPE = "neck_head"
PHASE2_SERVER_TRAIN_SCOPE = "all"
PHASE2_CLIENT_LR0_START = 1.2e-4
PHASE2_CLIENT_LR0_END = 2.0e-5
PHASE2_SERVER_LR0_START = 3.0e-4
PHASE2_SERVER_LR0_END = 1.0e-4
PHASE2_PSEUDO_LOSS_SCALE_START = 0.55
PHASE2_PSEUDO_LOSS_SCALE_END = 0.08
PHASE2_IGNORE_OBJ = True
MAX_PSEUDO_PER_IMAGE = 60
MAX_PSEUDO_PER_CLASS_IMAGE = 15

# Tri-stage pseudoGT policy inherited from 08.
CLIENT_LR0 = 3e-4
SERVER_LR0 = 1e-3
ADAPT_START_ROUND = 3
POLICY_HORIZON_ROUNDS = PHASE2_ROUNDS
LOW_MID_OBJ_WEIGHT = 0.45
MID_HIGH_OBJ_WEIGHT = 1.0

FORCE_WARMUP = False
FORCE_RESTART = False
APPEND_TRAIN_LOG = False
RUN_TRAINING = True
RUN_IN_BACKGROUND = True
STREAM_TRAIN_OUTPUT = False

try:
    import torch
    AVAILABLE_CUDA_GPUS = torch.cuda.device_count()
except Exception as exc:
    AVAILABLE_CUDA_GPUS = 0
    print("Could not inspect CUDA devices:", exc)

GPUS = min(REQUESTED_GPUS, AVAILABLE_CUDA_GPUS) if AVAILABLE_CUDA_GPUS else 1
if GPUS != REQUESTED_GPUS:
    print(f"Requested {REQUESTED_GPUS} GPU(s), visible={AVAILABLE_CUDA_GPUS}; using GPUS={GPUS}")


def find_free_port(preferred: int) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind(("127.0.0.1", preferred))
            return preferred
        except OSError:
            pass
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


MASTER_PORT = find_free_port(30142)

os.environ["DQA08_POLICY_MODEL"] = str(POLICY_MODEL)
os.environ["DQA08_POLICY_HORIZON_ROUNDS"] = str(POLICY_HORIZON_ROUNDS)
os.environ["DQA08_CLIENT_LR0"] = str(CLIENT_LR0)
os.environ["DQA08_SERVER_LR0"] = str(SERVER_LR0)
os.environ["DQA08_ADAPT_START_ROUND"] = str(ADAPT_START_ROUND)
os.environ["DQA08_LOW_MID_OBJ_WEIGHT"] = str(LOW_MID_OBJ_WEIGHT)
os.environ["DQA08_MID_HIGH_OBJ_WEIGHT"] = str(MID_HIGH_OBJ_WEIGHT)
os.environ["DQA08_THRESHOLD_LOG"] = str(THRESHOLD_LOG)

os.environ["DQA084_PHASE1_RESIDUAL_START"] = str(PHASE1_RESIDUAL_START)
os.environ["DQA084_PHASE1_RESIDUAL_END"] = str(PHASE1_RESIDUAL_END)
os.environ["DQA084_PHASE1_MAX_RELATIVE_UPDATE"] = str(PHASE1_MAX_RELATIVE_UPDATE)
os.environ["DQA084_PHASE1_SERVER_ANCHOR"] = str(PHASE1_SERVER_ANCHOR)
os.environ["DQA084_PHASE1_TEMPERATURE"] = str(PHASE1_TEMPERATURE)
os.environ["DQA084_PHASE1_STABILITY_LAMBDA"] = str(PHASE1_STABILITY_LAMBDA)
os.environ["DQA084_PHASE1_UNIFORM_MIX"] = str(PHASE1_UNIFORM_MIX)
os.environ["DQA084_PHASE1_ROUNDS"] = str(PHASE1_ROUNDS)

os.environ["DQA0843_PHASE2_RESIDUAL_START"] = str(PHASE2_RESIDUAL_START)
os.environ["DQA0843_PHASE2_RESIDUAL_END"] = str(PHASE2_RESIDUAL_END)
os.environ["DQA0843_PHASE2_CLASSWISE_BLEND_START"] = str(PHASE2_CLASSWISE_BLEND_START)
os.environ["DQA0843_PHASE2_CLASSWISE_BLEND_END"] = str(PHASE2_CLASSWISE_BLEND_END)
os.environ["DQA0843_PHASE2_MIN_SERVER_ALPHA_START"] = str(PHASE2_MIN_SERVER_ALPHA_START)
os.environ["DQA0843_PHASE2_MIN_SERVER_ALPHA_END"] = str(PHASE2_MIN_SERVER_ALPHA_END)
os.environ["DQA0843_PHASE2_SERVER_ANCHOR_START"] = str(PHASE2_SERVER_ANCHOR_START)
os.environ["DQA0843_PHASE2_SERVER_ANCHOR_END"] = str(PHASE2_SERVER_ANCHOR_END)
os.environ["DQA0843_PHASE2_TEMPERATURE_START"] = str(PHASE2_TEMPERATURE_START)
os.environ["DQA0843_PHASE2_TEMPERATURE_END"] = str(PHASE2_TEMPERATURE_END)
os.environ["DQA0843_PHASE2_STABILITY_LAMBDA_START"] = str(PHASE2_STABILITY_LAMBDA_START)
os.environ["DQA0843_PHASE2_STABILITY_LAMBDA_END"] = str(PHASE2_STABILITY_LAMBDA_END)
os.environ["DQA0843_PHASE2_DECAY_ROUNDS"] = str(PHASE2_DECAY_ROUNDS)
os.environ["DQA0843_PHASE2_CLIENT_TRAIN_SCOPE"] = PHASE2_CLIENT_TRAIN_SCOPE
os.environ["DQA0843_PHASE2_SERVER_TRAIN_SCOPE"] = PHASE2_SERVER_TRAIN_SCOPE
os.environ["DQA0843_PHASE2_CLIENT_LR0_START"] = str(PHASE2_CLIENT_LR0_START)
os.environ["DQA0843_PHASE2_CLIENT_LR0_END"] = str(PHASE2_CLIENT_LR0_END)
os.environ["DQA0843_PHASE2_SERVER_LR0_START"] = str(PHASE2_SERVER_LR0_START)
os.environ["DQA0843_PHASE2_SERVER_LR0_END"] = str(PHASE2_SERVER_LR0_END)
os.environ["DQA0843_PHASE2_PSEUDO_LOSS_SCALE_START"] = str(PHASE2_PSEUDO_LOSS_SCALE_START)
os.environ["DQA0843_PHASE2_PSEUDO_LOSS_SCALE_END"] = str(PHASE2_PSEUDO_LOSS_SCALE_END)
os.environ["DQA0843_PHASE2_IGNORE_OBJ"] = "1" if PHASE2_IGNORE_OBJ else "0"
os.environ["DQA0843_MAX_PSEUDO_PER_IMAGE"] = str(MAX_PSEUDO_PER_IMAGE)
os.environ["DQA0843_MAX_PSEUDO_PER_CLASS_IMAGE"] = str(MAX_PSEUDO_PER_CLASS_IMAGE)
os.environ.setdefault("ET_SKIP_AFTER_TRAIN_BEST_VAL", "1")

print("MASTER_PORT:", MASTER_PORT)
print("GPUS:", GPUS)


## Dry Run

設定ファイルとデータリスト生成だけを確認する。長い学習はまだ走らない。


In [ ]:
base_cmd = [
    str(PYTHON_BIN),
    str(RUN_SCRIPT),
    "--workspace-root", str(WORK_ROOT),
    "--stats-root", str(STATS_ROOT),
    "--warmup-epochs", str(WARMUP_EPOCHS),
    "--phase1-rounds", str(PHASE1_ROUNDS),
    "--phase2-rounds", str(PHASE2_ROUNDS),
    "--dqa-start-phase", str(DQA_START_PHASE),
    "--batch-size", str(BATCH_SIZE),
    "--workers", str(WORKERS),
    "--gpus", str(GPUS),
    "--master-port", str(MASTER_PORT),
    "--min-free-gib", str(MIN_FREE_GIB),
    "--classwise-blend", str(CLASSWISE_BLEND),
    "--server-anchor", str(SERVER_ANCHOR),
    "--temperature", str(TEMPERATURE),
    "--stability-lambda", str(STABILITY_LAMBDA),
    "--localize-bn",
    "--enable-dqa-guard",
    "--dqa-drop-ratio-threshold", "0.15",
    "--dqa-spike-ratio-threshold", "3.0",
]

dry_cmd = [*base_cmd[:2], "--dry-run", *base_cmd[2:]]
print(" ".join(dry_cmd))
subprocess.run(dry_cmd, cwd=REPO_ROOT, check=True, env=os.environ.copy())


## Train

`RUN_IN_BACKGROUND=True` のときはPIDとlogだけ残してバックグラウンドで走る。途中確認は次のStatusセルを再実行する。


In [ ]:
def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text(encoding="utf-8").strip())
    except ValueError:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "missing"
    result = subprocess.run(["ps", "-o", "stat=", "-p", str(pid)], capture_output=True, text=True)
    state = result.stdout.strip()
    if result.returncode != 0 or not state:
        return "missing"
    if "Z" in state:
        return "zombie"
    return state


train_cmd = [
    *base_cmd,
    "--log-file", str(TRAIN_LOG),
]
if FORCE_WARMUP:
    train_cmd.append("--force-warmup")
if FORCE_RESTART:
    train_cmd.append("--force-restart")
if APPEND_TRAIN_LOG:
    train_cmd.append("--append-train-log")
if STREAM_TRAIN_OUTPUT:
    train_cmd.append("--stream-train-output")

current_pid = read_pid(PID_PATH)
state = pid_state(current_pid)
print("existing pid:", current_pid, state)
print("command:", " ".join(train_cmd))

if RUN_TRAINING and state not in {"missing", "zombie"}:
    print("Training already appears to be running.")
elif RUN_TRAINING and RUN_IN_BACKGROUND:
    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_mode = "ab" if APPEND_TRAIN_LOG else "wb"
    with RUNNER_LOG.open(log_mode) as out:
        process = subprocess.Popen(
            train_cmd,
            cwd=REPO_ROOT,
            stdout=out,
            stderr=subprocess.STDOUT,
            env=os.environ.copy(),
            start_new_session=True,
        )
    PID_PATH.write_text(str(process.pid), encoding="utf-8")
    print("Started PID:", process.pid)
    print("Runner log:", RUNNER_LOG)
    print("Train log:", TRAIN_LOG)
elif RUN_TRAINING:
    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    with RUNNER_LOG.open("a", encoding="utf-8", buffering=1) as out:
        process = subprocess.Popen(
            train_cmd,
            cwd=REPO_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=os.environ.copy(),
        )
        PID_PATH.write_text(str(process.pid), encoding="utf-8")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            out.write(line)
        raise SystemExit(process.wait())
else:
    print("RUN_TRAINING=False, not starting.")


In [ ]:
# 起動通知。完了通知ではない。
try:
    sys.path.insert(0, str(REPO_ROOT))
    from notebook_notify import notify_discord
    result = notify_discord(
        "08_4_3 DQA-SBA short plateau phase2 runを起動しました",
        title="実行開始",
        context={"workspace": str(WORK_ROOT), "pid": read_pid(PID_PATH), "runner_log": str(RUNNER_LOG)},
        fail_silently=True,
    )
    print(result)
except Exception as exc:
    print("notify skipped:", exc)


## Status

学習中はこのセルだけ再実行する。Phase 1のDQA-SBA重みは `dqa_cwa_state.json` の `phase1_dqa_sba`、Phase 2のplateau scheduleは `phase2_stable_plateau` に記録される。


In [ ]:
pid = read_pid(PID_PATH)
print("pid:", pid, pid_state(pid))
print("workspace exists:", WORK_ROOT.exists())
print("history:", WORK_ROOT / "history.json", (WORK_ROOT / "history.json").exists())

if (WORK_ROOT / "history.json").exists():
    history = json.loads((WORK_ROOT / "history.json").read_text(encoding="utf-8"))
    print("completed rounds:", len(history))
    print(history[-5:])

state_path = WORK_ROOT / "dqa_cwa_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    phase1_records = state.get("phase1_dqa_sba", [])
    print("phase1_dqa_sba records:", len(phase1_records))
    if phase1_records:
        print(json.dumps(phase1_records[-1], indent=2)[:2200])
    phase2_records = state.get("phase2_stable_plateau", [])
    print("phase2_stable_plateau records:", len(phase2_records))
    if phase2_records:
        print(json.dumps(phase2_records[-1], indent=2)[:2200])

for label, path in [("runner", RUNNER_LOG), ("train", TRAIN_LOG)]:
    print("\n---", label, path)
    if path.exists():
        result = subprocess.run(["tail", "-40", str(path)], capture_output=True, text=True)
        print(result.stdout[-5000:])
    else:
        print("missing")


## Evaluate

学習完了後に実行する。trendを見るため、warmup、Phase 1中間/最終、Phase 2前半/中盤/最終をscene splitで評価する。


In [ ]:
history_path = WORK_ROOT / "history.json"
global_dir = WORK_ROOT / "global_checkpoints"
checkpoint_specs = []
for label, rel in [
    ("warmup_global", "round000_warmup.pt"),
    ("phase1_round003_global", "phase1_round003_global.pt"),
    ("phase1_round006_global", "phase1_round006_global.pt"),
    ("phase2_round001_global", "phase2_round001_global.pt"),
    ("phase2_round002_global", "phase2_round002_global.pt"),
    ("phase2_round003_global", "phase2_round003_global.pt"),
    ("phase2_round004_global", "phase2_round004_global.pt"),
    ("phase2_round008_global", "phase2_round008_global.pt"),
    ("phase2_round012_global", "phase2_round012_global.pt"),
    ("phase2_round016_global", "phase2_round016_global.pt"),
]:
    path = global_dir / rel
    if path.exists():
        checkpoint_specs.append((label, path))

if not checkpoint_specs:
    print("No completed checkpoint yet. Training is probably still in warmup/early rounds; run Status again later, then evaluate.")
else:
    print("selected checkpoints:", len(checkpoint_specs))
    for label, path in checkpoint_specs:
        print(label, path)
    eval_cmd = [
        str(PYTHON_BIN),
        str(EVAL_SCRIPT),
        "--workspace", str(WORK_ROOT),
        "--batch-size", "16",
        "--device", "0" if GPUS >= 1 else "",
        "--no-plots",
    ]
    for label, path in checkpoint_specs:
        eval_cmd.extend(["--checkpoint", f"{label}={path}"])
    print(" ".join(eval_cmd))
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)

    summary = WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"
    if summary.exists():
        df = pd.read_csv(summary)
        display(df)
        pivot = df.pivot_table(index="checkpoint_label", columns="split", values=["map50", "map50_95"], aggfunc="first")
        display(pivot)
    else:
        print("missing summary:", summary)


In [ ]:
# 評価完了後の通知。
try:
    sys.path.insert(0, str(REPO_ROOT))
    from notebook_notify import notify_discord
    summary = WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"
    if summary.exists():
        df = pd.read_csv(summary)
        msg = df.to_string(index=False)
    else:
        msg = "08_4_3 evaluation cell finished, but summary CSV was not found."
    result = notify_discord(
        msg,
        title="08_4_3 評価完了",
        context={"workspace": str(WORK_ROOT), "summary": str(summary)},
        fail_silently=True,
    )
    print(result)
except Exception as exc:
    print("notify skipped:", exc)


## 実行結果メモ

2026-05-04 UTCに実行完了。学習時間は約2時間54分、評価込みでも4時間以内。

`scene_total`:

| checkpoint | mAP@0.5 | mAP@0.5:0.95 | P | R |
| --- | ---: | ---: | ---: | ---: |
| warmup | 0.349 | 0.187 | 0.503 | 0.350 |
| Phase1 r003 | 0.379 | 0.205 | 0.469 | 0.415 |
| Phase1 r006 | 0.377 | 0.203 | 0.467 | 0.413 |
| Phase2 r001 | 0.377 | 0.208 | 0.477 | 0.408 |
| Phase2 r004 | 0.376 | 0.208 | 0.482 | 0.407 |
| Phase2 r008 | 0.376 | 0.208 | 0.485 | 0.406 |
| Phase2 r012 | 0.376 | 0.208 | 0.483 | 0.408 |
| Phase2 r016 | 0.375 | 0.207 | 0.486 | 0.405 |

08_4_2では Phase2 r012 が `mAP50=0.356 / mAP50-95=0.194` まで落ちたが、08_4_3では r001-r016 が `mAP50-95=0.207-0.208` にほぼ固定された。Phase2を何roundも回しても性能が崩れない、という目的にはかなり近い。

ただし pseudo stats は Phase2後半でも mean confidence/objectness が上がっている。今回の改善は「品質推定が正しくなった」というより、短期impulse後に residual/classwise/pseudo loss/LR をほぼゼロに落としたことで、過信したpseudo boxがモデルを動かす力を失った、という解釈が自然。
